In [0]:
# Gold Layer: Inventory Health & Stockout Risk

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, sum as spark_sum, count, when, 
    round as spark_round, current_timestamp
)

spark = SparkSession.builder.getOrCreate()
GOLD_TABLE = "ecomstore.ecomlake.gold_inventory_health"

inventory = spark.read.table("ecomstore.ecomlake.silver_inventory")
products = spark.read.table("ecomstore.ecomlake.silver_products")

# 1. ENRICH INVENTORY WITH PRODUCT METADATA
enriched_inventory = (
    inventory.alias("i")
    .join(products.alias("p"), on="product_id", how="inner")
)

# 2. AGGREGATE HEALTH BY WAREHOUSE (DARK STORE) & CATEGORY
inventory_health = (
    enriched_inventory
    .groupBy("i.warehouse_id", "p.category")
    .agg(
        count("i.product_id").alias("total_unique_skus"),
        spark_sum("i.stock_quantity").alias("total_physical_stock"),
        spark_sum("i.reserved_quantity").alias("total_reserved_stock"),
        spark_sum("i.available_quantity").alias("total_sellable_stock"),
        
        # Risk Metrics
        count(when(col("i.is_below_reorder_level") == True, True)).alias("skus_below_reorder_level"),
        count(when(col("i.available_quantity") == 0, True)).alias("out_of_stock_skus")
    )
    .withColumn("stockout_risk_pct", 
        when(col("total_unique_skus") > 0, 
             spark_round(col("skus_below_reorder_level") / col("total_unique_skus") * 100, 2))
        .otherwise(0.0)
    )
    .withColumn("_gold_processed_at", current_timestamp())
    .orderBy("warehouse_id", col("stockout_risk_pct").desc())
)

# 3. WRITE TO GOLD
(
    inventory_health
    .coalesce(1)
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_TABLE)
)